# 04 群聚調查工作流 — 練習

用松柏護理之家退伍軍人症 line list，自己走一遍 SitRep 產出流程。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## 題目 1：摘要指標

1. 讀入 `data/synthetic/legionella_outbreak.csv`
2. 建立 `infected` 欄位（`clinical_severity != 'not_ill'`）
3. 計算並印出：住民總數、感染人數、侵襲率、確診數、可能數、住院數、ICU 數、死亡數、CFR

In [ ]:
# TODO: 讀入 CSV 並建立 infected 欄位
# TODO: 計算所有摘要指標
# TODO: 印出結構化的 SitRep 摘要

## 題目 2：人時地三要素

1. **人**：計算感染者的年齡中位數、男性比例
2. **時**：畫一張流行曲線（matplotlib bar chart），標出流行期間和高峰日
3. **地**：用 `groupby(["floor", "wing"])` 計算各翼區侵襲率，印出表格

In [ ]:
# TODO: 人 (Person) — 年齡中位數、男性比例

In [ ]:
# TODO: 時 (Time) — 流行曲線 + 流行期間 + 高峰日

In [ ]:
# TODO: 地 (Place) — 翼區侵襲率表

## 題目 3：按年齡組的分層摘要

1. 建立 `age_group` 欄位（60-69 / 70-79 / 80-89 / 90+）
2. 用 `groupby("age_group")` 計算各年齡組的：人數、感染數、侵襲率、死亡數、CFR
3. 哪個年齡組的侵襲率最高？哪個年齡組的 CFR 最高？兩者一樣嗎？

In [ ]:
# TODO: 建立 age_group
# TODO: groupby + agg
# TODO: 計算 AR% 和 CFR%
# TODO: 印出表格並回答問題

## 題目 4（挑戰題）：寫一個 generate_sitrep 函式

把題目 1-3 的邏輯包成一個函式 `generate_sitrep(csv_path)`，回傳一個 dict，包含：
- `total_residents`, `infected`, `attack_rate`, `deaths`, `cfr`, `hospitalized`, `icu`
- `peak_date`（高峰日）
- `worst_wing`（侵襲率最高的翼區名稱）

呼叫函式並印出結果。

In [ ]:
# TODO: 定義 generate_sitrep(csv_path) 函式
# TODO: 呼叫並印出結果

## 題目 5：產出一份 Word 報告

利用 `python-docx` 套件，將 SitRep 的摘要指標和流行曲線匯出為一份 `.docx` 文件。

1. 建立一份 Word 文件，標題為「松柏護理之家退伍軍人症 SitRep」
2. 加入報告時間（`datetime.now()`）
3. 加入一個摘要指標表格（住民總數、感染人數、侵襲率、死亡數、CFR）
4. 將流行曲線圖嵌入文件中（提示：先用 `BytesIO` 將 matplotlib 圖表存成 PNG，再用 `doc.add_picture()`）
5. 儲存到 `output/my_sitrep.docx`

提示：套件安裝 `pip install python-docx`，匯入時用 `from docx import Document`。

In [ ]:
# TODO: from docx import Document
# TODO: from docx.shared import Inches
# TODO: 建立 Document, add_heading, add_paragraph
# TODO: 建立摘要指標表格 (add_table)
# TODO: 用 BytesIO 存流行曲線 → doc.add_picture()
# TODO: doc.save("output/my_sitrep.docx")

## 題目 6：食因性群聚的暴露分析（諾羅病毒情境）

某社區辦桌宴會後爆發諾羅病毒群聚，下方資料是賓客名冊（`ate_oysters` 標示是否食用生蠔冷盤）。

1. 建立 `ate_oysters`（是否食用生蠔冷盤）兩組的暴露表：各組人數、病例數
2. 計算兩組的侵襲率，並用 `risk_ratio()` 計算風險比 RR
3. 依 `symptom_onset_date` 畫出流行曲線（可用 `plot_epi_curve()` 或自行 `groupby`）
4. 印出摘要：賓客總數、病例數、整體侵襲率、RR
5. 從流行曲線的形狀（單一高峰、快速下降）判斷：這是點源型（point source）還是連續共同源（continuous common source）群聚？RR 的結果是否支持生蠔冷盤為可疑暴露來源？

In [ ]:
import numpy as np

# 資料：某社區辦桌宴會後的賓客名冊（模擬諾羅病毒食因性群聚）
rng = np.random.default_rng(614)
n = 240
banquet_date = pd.Timestamp("2026-03-14")

guest_id = np.arange(1, n + 1)
table_no = rng.integers(1, 25, n)  # 24 桌
ate_oysters = rng.random(n) < 0.4  # 4 成賓客食用生蠔冷盤

# 食用生蠔冷盤者感染機率明顯較高（可疑暴露）
p_infect = np.where(ate_oysters, 0.65, 0.08)
infected = rng.random(n) < p_infect

# 諾羅病毒潛伏期短（約 12-48 小時），只有病例才有發病日
incubation_hours = rng.normal(30, 8, n).clip(10, 60)
onset_datetime = pd.DatetimeIndex(banquet_date + pd.to_timedelta(incubation_hours, unit="h"))

df6 = pd.DataFrame({
    "guest_id": guest_id,
    "table_no": table_no,
    "ate_oysters": np.where(ate_oysters, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df6.loc[infected, "symptom_onset_date"] = onset_datetime[infected].normalize()

# TODO: 建立 ate_oysters 兩組的暴露表（各組人數、病例數）
# TODO: 計算兩組侵襲率，並用 risk_ratio() 計算 RR
# TODO: 依 symptom_onset_date 畫流行曲線（plot_epi_curve 或 groupby）
# TODO: 印出摘要（賓客總數、病例數、整體侵襲率、RR）
# TODO: 回答：這是點源型還是連續共同源群聚？生蠔冷盤是否為可疑暴露來源？

## 題目 7：跨部門群聚調查（COVID-19 情境）

某公司於 3/1 舉辦部門聚餐後爆發 COVID-19 群聚，下方資料是員工名冊（`attended_meeting` 標示是否參加聚餐）。

1. 用 `summarize_by_group()` 依 `department` 摘要病例數與佔比
2. 計算各部門的侵襲率（各部門感染人數 / 各部門總人數），排序找出風險最高的部門
3. 依 `symptom_onset_date` 用 `plot_epi_curve()` 畫出流行曲線
4. 計算參加聚餐者 vs 未參加者的風險比 RR
5. 哪個部門侵襲率最高？是否與該部門的聚餐參加率有關？RR 的結果是否支持聚餐是本次群聚的傳播熱點？

In [ ]:
import numpy as np

# 資料：某公司部門聚餐後的員工名冊（模擬 COVID-19 職場群聚）
rng = np.random.default_rng(719)
n = 450
departments = ["業務部", "行政部", "IT部", "財務部", "客服部"]
dept_sizes = [150, 80, 70, 60, 90]
department = np.repeat(departments, dept_sizes)

# 各部門參加聚餐的比例不同（業務部聚餐參加率最高）
p_meeting = {"業務部": 0.75, "行政部": 0.30, "IT部": 0.15, "財務部": 0.20, "客服部": 0.35}
attended_meeting = np.array([rng.random() < p_meeting[d] for d in department])

# 參加聚餐者感染機率明顯較高
p_infect = np.where(attended_meeting, 0.55, 0.05)
infected = rng.random(n) < p_infect

# COVID-19 潛伏期約 2-8 天
onset_offset_days = rng.integers(2, 9, n)
meeting_date = pd.Timestamp("2026-03-01")
onset_date = meeting_date + pd.to_timedelta(onset_offset_days, unit="D")

df7 = pd.DataFrame({
    "employee_id": np.arange(1, n + 1),
    "department": department,
    "attended_meeting": np.where(attended_meeting, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df7.loc[infected, "symptom_onset_date"] = onset_date[infected]

# TODO: 用 summarize_by_group() 依 department 摘要病例數與佔比（只取感染者子集）
# TODO: 用 groupby("department") 計算各部門侵襲率，依侵襲率排序
# TODO: 依 symptom_onset_date 用 plot_epi_curve() 畫流行曲線
# TODO: 計算 attended_meeting 是否參加聚餐兩組的風險比 RR
# TODO: 回答：哪個部門侵襲率最高？是否與聚餐參加率有關？RR 是否支持聚餐是傳播熱點？

## 題目 8（挑戰題）：疫苗覆蓋率與世代間隔（麻疹校園群聚情境）

某國小爆發麻疹群聚，下方資料模擬班級內的傳播鏈：每個病例記錄了感染源 `infector_id` 與所屬世代 `generation`（世代 0 為社區感染的指標病例）。

1. 計算已接種 vs 未接種疫苗學生的侵襲率，並用 `risk_ratio()` 計算風險比 RR
2. 依 `symptom_onset_date` 畫出流行曲線，觀察病例是否分成多個波段（世代）
3. 對每個有 `infector_id` 的續發病例，計算其發病日與感染源發病日之差，估計世代間隔（serial interval）的平均值與中位數
4. 印出 SitRep 摘要（學生總數、病例數、侵襲率、RR、世代間隔估計值）
5. 回答：估計出的世代間隔是否接近文獻報告的麻疹世代間隔（約 11–12 天）？此校 92% 的疫苗覆蓋率是否達到麻疹的群體免疫閾值（約 95%）？這與群聚能持續擴散有何關係？

In [ ]:
import numpy as np

# 資料：某國小麻疹群聚，模擬班級內的傳播鏈（含世代與感染源）
rng = np.random.default_rng(2026)
n_classes = 20
students_per_class = 20
n = n_classes * students_per_class  # 400 名學生

class_id = np.repeat([f"C{i+1:02d}" for i in range(n_classes)], students_per_class)
student_id = np.arange(1, n + 1)
vaccinated = rng.random(n) < 0.92  # 疫苗覆蓋率 92%（低於麻疹群體免疫閾值約 95%）

df8 = pd.DataFrame({
    "student_id": student_id,
    "class_id": class_id,
    "vaccinated": np.where(vaccinated, "yes", "no"),
})
df8["infected"] = False
df8["symptom_onset_date"] = pd.NaT
df8["infector_id"] = pd.NA
df8["generation"] = pd.NA

start_date = pd.Timestamp("2026-03-02")

# 世代 0：3 名社區感染的未接種指標病例
unvacc_idx = df8.index[df8["vaccinated"] == "no"].to_numpy()
primary_idx = rng.choice(unvacc_idx, size=3, replace=False)
for idx in primary_idx:
    df8.loc[idx, "infected"] = True
    df8.loc[idx, "symptom_onset_date"] = start_date + pd.Timedelta(days=int(rng.integers(0, 3)))
    df8.loc[idx, "generation"] = 0

# 依世代模擬班級內傳播鏈：每個病例可能傳染同班未感染的同學，
# 已接種者仍可能被感染，但機率因疫苗保護力（97%）大幅降低
vaccine_efficacy = 0.97
p_transmit_unvacc = 0.55  # 每一對同班接觸者間的傳染機率（未接種）
max_generations = 5

current_gen = 0
while current_gen < max_generations:
    infectors = df8[(df8["generation"] == current_gen) & df8["infected"]]
    if infectors.empty:
        break
    for _, case in infectors.iterrows():
        classmates = df8[
            (df8["class_id"] == case["class_id"])
            & (~df8["infected"])
            & (df8.index != case.name)
        ]
        for cm_idx, cm in classmates.iterrows():
            transmit_prob = (
                p_transmit_unvacc * (1 - vaccine_efficacy)
                if cm["vaccinated"] == "yes"
                else p_transmit_unvacc
            )
            if rng.random() < transmit_prob:
                generation_interval = max(7, rng.normal(12, 2))  # 世代間隔天數
                onset = case["symptom_onset_date"] + pd.Timedelta(days=generation_interval)
                df8.loc[cm_idx, "infected"] = True
                df8.loc[cm_idx, "symptom_onset_date"] = onset
                df8.loc[cm_idx, "infector_id"] = case["student_id"]
                df8.loc[cm_idx, "generation"] = current_gen + 1
    current_gen += 1

# TODO: 計算未接種 vs 已接種學生的侵襲率，並用 risk_ratio() 計算 RR
# TODO: 依 symptom_onset_date 畫流行曲線（plot_epi_curve），觀察是否有多個世代波段
# TODO: 對每個有 infector_id 的病例，計算其發病日與感染源發病日之差 -> 估計世代間隔平均值與中位數
# TODO: 印出 SitRep 摘要（學生總數、病例數、侵襲率、RR、世代間隔估計值）
# TODO: 回答：世代間隔是否接近文獻的 11-12 天？92% 疫苗覆蓋率是否達到群體免疫閾值？